# 🤖 EKSTRAKSI SENTIMEN INDOBERT — BAB 4.1.3
## Tesis: Pemodelan Prediksi Volatilitas Return Saham Sektor Energi LQ45
**Mahasiswa :** Ilda Nurida (NRP 6032251068)
**Prodi      :** Magister Manajemen Teknologi — SIMT ITS
**Tahun      :** 2026

---
### 🗂️ Alur Notebook Ini
| Cell | Proses | Output |
|------|--------|--------|
| 1 | Konfigurasi & batasan penelitian | — |
| 2 | Load dataset berita + filter emiten & periode | — |
| 3 | Pembersihan teks (boilerplate situs) | — |
| 4 | Klasifikasi sentimen IndoBERT (batch, GPU-aware) | — |
| 5 | Tabel distribusi sentimen (→ Tabel 4.X) | `tabel_4_x_distribusi_sentimen.csv` |
| 6 | Agregasi harian per emiten (fitur X2) | `sentimen_harian_per_emiten.csv` |
| 7 | Visualisasi distribusi & deret waktu | `gambar_4_x_*.png` |
| 8 | Simpan skor per artikel | `sentimen_artikel_FINAL.csv` |

**Input** : `berita_gabungan_semua_sumber_FINAL.csv` (output GABUNG_BERITA_FINAL.ipynb)
**Output** dipakai oleh PREP sentimen → MODEL_xgboost / MODEL_lstm
(fitur: `Sentiment_Mean`, `Sentiment_Dispersion`, `News_Count`, `Ada_Berita`)


---
## ⚙️ CELL 1 — Import, Konfigurasi, dan Batasan Penelitian

In [ ]:
# !pip install transformers torch sastrawi -q   # jalankan sekali bila belum terpasang
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
plt.rcParams.update({'font.family': 'Times New Roman', 'font.size': 11, 'figure.dpi': 150})

# ── ROOT auto-detect (pola sama dengan MODEL_xgboost_vv2) ──
ROOT = Path(os.getcwd())
for _ in range(6):
    if (ROOT / '03_dataset').exists(): break
    ROOT = ROOT.parent
if not (ROOT / '03_dataset').exists():
    ROOT = Path(r'e:\DATA PENTING ILDA NURIDA\DOKUMEN\KULIAH\S2 2025\THESIS'
                r'\Penelitian\TESIS_PEMODELAN PREDIKSI VOLATILITAS RETURN SAHAM_2026')

DIR_FINAL  = ROOT / '03_dataset' / 'final'
DIR_TABEL  = ROOT / '06_dokumen_tesis' / 'tabel'
DIR_GAMBAR = ROOT / '06_dokumen_tesis' / 'gambar' / 'sentimen'
for d in (DIR_TABEL, DIR_GAMBAR): d.mkdir(parents=True, exist_ok=True)

PATH_BERITA = DIR_FINAL / 'berita_gabungan_semua_sumber_FINAL.csv'

# ── BATASAN PENELITIAN (Bab 3) ──
# 6 emiten LQ45 sektor energi; PGAS di luar cakupan pemodelan
EMITEN_PENELITIAN = ['ADRO', 'ADMR', 'ITMG', 'PTBA', 'MEDC', 'AKRA']
PERIODE_AWAL  = '2023-04-01'
PERIODE_AKHIR = '2026-05-31'

# ── KONFIGURASI MODEL SENTIMEN ──
# CATATAN METODOLOGIS: 'indobenchmark/indobert-base-p2' adalah model PRA-LATIH
# (masked language model) TANPA head klasifikasi sentimen. Untuk klasifikasi
# 3 kelas digunakan checkpoint IndoBERT yang telah di-FINE-TUNE pada korpus
# sentimen Bahasa Indonesia. Di naskah tesis, tuliskan: "IndoBERT (Wilie et
# al., 2020) yang telah dilatih-halus (fine-tuned) untuk klasifikasi sentimen
# tiga kelas". Arsitektur dasarnya tetap IndoBERT-base.
MODEL_CHECKPOINT = 'mdhugol/indonesia-bert-sentiment-classification'
# mapping label checkpoint di atas → skor numerik penelitian
LABEL2SKOR = {'LABEL_0': 1, 'LABEL_1': 0, 'LABEL_2': -1}   # positif, netral, negatif
NAMA_KELAS = {1: 'Positif', 0: 'Netral', -1: 'Negatif'}

MAX_LENGTH   = 256      # token; headline + lead paragraf sudah tercakup
BATCH_SIZE   = 32       # turunkan ke 8-16 bila hanya CPU
PANJANG_ISI  = 600      # karakter awal Isi_Berita yang diikutkan (lead)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('=' * 70)
print('✅ KONFIGURASI EKSTRAKSI SENTIMEN')
print('=' * 70)
print(f'ROOT      : {ROOT}')
print(f'Input     : {PATH_BERITA.name}')
print(f'Emiten    : {EMITEN_PENELITIAN}')
print(f'Periode   : {PERIODE_AWAL} s.d. {PERIODE_AKHIR}')
print(f'Model     : {MODEL_CHECKPOINT}')
print(f'Device    : {DEVICE.upper()} | Batch: {BATCH_SIZE} | Max token: {MAX_LENGTH}')
print('=' * 70)

---
## 📂 CELL 2 — Load Dataset Berita + Filter Batasan Penelitian

In [ ]:
df_raw = pd.read_csv(PATH_BERITA)
df_raw['Tanggal'] = pd.to_datetime(df_raw['Tanggal'], errors='coerce')

n0 = len(df_raw)
df = df_raw.dropna(subset=['Tanggal', 'Headline']).copy()

# Filter 1: emiten sesuai batasan penelitian (PGAS dikeluarkan)
df = df[df['Kode_Emiten'].isin(EMITEN_PENELITIAN)]
n1 = len(df)

# Filter 2: periode penelitian
df = df[(df['Tanggal'] >= PERIODE_AWAL) & (df['Tanggal'] <= PERIODE_AKHIR)]
n2 = len(df)

df = df.sort_values(['Kode_Emiten', 'Tanggal']).reset_index(drop=True)

print('=' * 70)
print('  📂 FILTER DATASET SESUAI BATASAN PENELITIAN')
print('=' * 70)
print(f'  Artikel awal (semua emiten & periode) : {n0:>6,}')
print(f'  Setelah filter 6 emiten (tanpa PGAS)  : {n1:>6,}  (-{n0-n1:,})')
print(f'  Setelah filter periode Apr23–Mei26    : {n2:>6,}  (-{n1-n2:,})')
print()
print(df.groupby('Kode_Emiten')['Headline'].count()
        .reindex(EMITEN_PENELITIAN).rename('Jumlah Artikel').to_frame())

---
## 🧹 CELL 3 — Pembersihan Teks (Boilerplate Situs & Normalisasi)

In [ ]:
# Pola boilerplate/disclaimer yang teridentifikasi dari word cloud (Gambar 4.4):
# frasa hak cipta & navigasi situs yang ikut ter-scrape dan BUKAN isi berita.
# Frasa 'dilarang keras' berkonotasi negatif → wajib dibuang agar skor
# sentimen IndoBERT tidak bias.
POLA_BOILERPLATE = [
    r'dilarang keras mengambil konten[^.]*\.?',
    r'(konten\s+)?melakukan crawling[^.]*\.?',
    r'pengindeksan otomatis[^.]*\.?',
    r'(tanpa\s+)?izin tertulis[^.]*\.?',
    r'mengambil konten[^.]*\.?',
    r'di\s?google news', r'lain di\s?google', r'digoogle news',
    r'baca juga\s*:[^.]*\.?',
    r'simak (video|berita)[^.]*\.?',
    r'advertisement|advertorial|scroll to continue with content',
]
RE_BOILER = [re.compile(p, flags=re.IGNORECASE) for p in POLA_BOILERPLATE]

def bersihkan_teks(teks):
    t = str(teks) if pd.notna(teks) else ''
    for rgx in RE_BOILER:
        t = rgx.sub(' ', t)
    t = re.sub(r'http\S+|www\.\S+', ' ', t)      # URL
    t = re.sub(r'\s+', ' ', t).strip()
    return t

# Teks input model = Headline + lead Isi_Berita (konteks kalimat utuh —
# stopword TIDAK dibuang karena transformer membutuhkan konteks lengkap)
df['Isi_Bersih'] = df['Isi_Berita'].apply(bersihkan_teks)
df['Teks_Model'] = (df['Headline'].astype(str).str.strip() + '. '
                    + df['Isi_Bersih'].str[:PANJANG_ISI]).str.strip()

# Audit efek pembersihan
contoh_kena = df[df['Isi_Berita'].astype(str).str.contains('dilarang keras', case=False, na=False)]
print('=' * 70)
print('  🧹 PEMBERSIHAN TEKS SELESAI')
print('=' * 70)
print(f'  Artikel mengandung boilerplate "dilarang keras" : {len(contoh_kena):,}')
print(f'  Rata-rata panjang Teks_Model                    : {df["Teks_Model"].str.len().mean():.0f} karakter')
if len(contoh_kena):
    i = contoh_kena.index[0]
    print('\n  Contoh SEBELUM :', str(df.loc[i, 'Isi_Berita'])[:160], '...')
    print('  Contoh SESUDAH :', df.loc[i, 'Teks_Model'][:160], '...')

---
## 🤖 CELL 4 — Klasifikasi Sentimen IndoBERT (Batch Inference)

In [ ]:
print(f'⏳ Memuat model {MODEL_CHECKPOINT} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT)
model.to(DEVICE).eval()
id2label = model.config.id2label
print(f'✅ Model siap di {DEVICE.upper()} | label: {id2label}')

@torch.no_grad()
def klasifikasi_batch(daftar_teks):
    enc = tokenizer(daftar_teks, padding=True, truncation=True,
                    max_length=MAX_LENGTH, return_tensors='pt').to(DEVICE)
    logits = model(**enc).logits
    prob = torch.softmax(logits, dim=-1).cpu().numpy()
    idx  = prob.argmax(axis=1)
    return idx, prob[np.arange(len(idx)), idx]

skor_list, conf_list = [], []
teks_semua = df['Teks_Model'].tolist()
n_batch = int(np.ceil(len(teks_semua) / BATCH_SIZE))

for b in range(n_batch):
    batch = teks_semua[b*BATCH_SIZE:(b+1)*BATCH_SIZE]
    idx, conf = klasifikasi_batch(batch)
    skor_list.extend(LABEL2SKOR[id2label[i]] for i in idx)
    conf_list.extend(conf)
    if (b+1) % 20 == 0 or (b+1) == n_batch:
        print(f'  Batch {b+1:>4}/{n_batch} | {min((b+1)*BATCH_SIZE, len(teks_semua)):,} artikel', end='\r')

df['Skor_Sentimen'] = skor_list
df['Label_Sentimen'] = df['Skor_Sentimen'].map(NAMA_KELAS)
df['Confidence'] = np.round(conf_list, 4)
print('\n✅ Klasifikasi selesai:', f'{len(df):,} artikel')

---
## 📋 CELL 5 — Tabel Distribusi Sentimen per Emiten (→ Tabel 4.X)

In [ ]:
tab_dist = (df.pivot_table(index='Kode_Emiten', columns='Label_Sentimen',
                           values='Headline', aggfunc='count', fill_value=0)
              .reindex(index=EMITEN_PENELITIAN,
                       columns=['Positif', 'Netral', 'Negatif'], fill_value=0))
tab_dist['Total'] = tab_dist.sum(axis=1)
for k in ['Positif', 'Netral', 'Negatif']:
    tab_dist[f'{k} (%)'] = (tab_dist[k] / tab_dist['Total'] * 100).round(1)
tab_dist.loc['TOTAL'] = tab_dist.sum(numeric_only=True)
for k in ['Positif', 'Netral', 'Negatif']:
    tab_dist.loc['TOTAL', f'{k} (%)'] = round(
        tab_dist.loc['TOTAL', k] / tab_dist.loc['TOTAL', 'Total'] * 100, 1)

print('=' * 70)
print('  📋 TABEL 4.X — DISTRIBUSI SENTIMEN PER EMITEN')
print('=' * 70)
print(tab_dist.astype({'Positif': int, 'Netral': int, 'Negatif': int, 'Total': int}))
print(f'\n  Rata-rata confidence model : {df["Confidence"].mean():.3f}')

path_tab = DIR_TABEL / 'tabel_4_x_distribusi_sentimen.csv'
tab_dist.to_csv(path_tab)
print(f'  💾 Disimpan: {path_tab}')

---
## 📅 CELL 6 — Agregasi Harian per Emiten (Fitur X2 untuk Model)

In [ ]:
# Menghasilkan fitur X2 PERSIS seperti yang diharapkan MODEL_xgboost/MODEL_lstm:
#   Sentiment_Mean       : rata-rata skor sentimen hari t
#   Sentiment_Dispersion : simpangan baku skor hari t (0 bila artikel < 2)
#   News_Count           : jumlah artikel hari t
#   Ada_Berita           : indikator biner ada/tidaknya berita hari t
# Hari tanpa berita diisi netral (mean=0, dispersion=0, count=0, ada=0) —
# keputusan ini konsisten dengan asumsi "tidak ada sinyal = netral".

kalender = pd.date_range(PERIODE_AWAL, PERIODE_AKHIR, freq='D')
list_harian = []
for emiten in EMITEN_PENELITIAN:
    d = df[df['Kode_Emiten'] == emiten]
    g = (d.groupby(d['Tanggal'].dt.normalize())['Skor_Sentimen']
           .agg(Sentiment_Mean='mean', Sentiment_Dispersion='std', News_Count='count'))
    g = g.reindex(kalender)
    g['News_Count'] = g['News_Count'].fillna(0).astype(int)
    g['Ada_Berita'] = (g['News_Count'] > 0).astype(int)
    g['Sentiment_Mean'] = g['Sentiment_Mean'].fillna(0.0)
    g['Sentiment_Dispersion'] = g['Sentiment_Dispersion'].fillna(0.0)
    g.insert(0, 'Kode_Emiten', emiten)
    g.index.name = 'Tanggal'
    list_harian.append(g.reset_index())

df_harian = pd.concat(list_harian, ignore_index=True)
df_harian['Sentiment_Mean'] = df_harian['Sentiment_Mean'].round(4)
df_harian['Sentiment_Dispersion'] = df_harian['Sentiment_Dispersion'].round(4)

print('=' * 70)
print('  📅 AGREGASI SENTIMEN HARIAN')
print('=' * 70)
print(f'  Baris: {len(df_harian):,} = {len(EMITEN_PENELITIAN)} emiten × {len(kalender):,} hari')
print(f'  Proporsi hari dengan berita: '
      f'{df_harian["Ada_Berita"].mean()*100:.1f}%')
print()
print(df_harian.groupby('Kode_Emiten')
      .agg(Hari_Berberita=('Ada_Berita', 'sum'),
           Rerata_Sentimen=('Sentiment_Mean', 'mean'),
           Rerata_Artikel_per_Hari=('News_Count', 'mean')).round(3))

---
## 📊 CELL 7 — Visualisasi Distribusi & Deret Waktu Sentimen (→ Gambar 4.X)

In [ ]:
WARNA_SENTIMEN = {'Positif': '#2d6a4f', 'Netral': '#adb5bd', 'Negatif': '#9d0208'}

# (a) Stacked bar distribusi per emiten
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5),
                               gridspec_kw={'width_ratios': [1, 1.5]})
bawah = np.zeros(len(EMITEN_PENELITIAN))
basis = tab_dist.loc[EMITEN_PENELITIAN]
for kelas in ['Positif', 'Netral', 'Negatif']:
    nilai = basis[kelas].values.astype(float)
    ax1.bar(EMITEN_PENELITIAN, nilai, bottom=bawah,
            label=kelas, color=WARNA_SENTIMEN[kelas], edgecolor='white', width=0.65)
    bawah += nilai
ax1.set_title('(a) Distribusi Kelas Sentimen per Emiten', fontweight='bold')
ax1.set_ylabel('Jumlah Artikel'); ax1.legend(frameon=False)
ax1.grid(axis='y', alpha=0.3, linestyle='--'); ax1.spines[['top', 'right']].set_visible(False)

# (b) Deret waktu rata-rata sentimen bulanan (agregat 6 emiten)
bulanan = (df.set_index('Tanggal').groupby(pd.Grouper(freq='M'))['Skor_Sentimen']
             .mean())
ax2.plot(bulanan.index, bulanan.values, color='#1d3557', linewidth=1.8)
ax2.axhline(0, color='#adb5bd', linewidth=0.8, linestyle='--')
ax2.fill_between(bulanan.index, bulanan.values, 0,
                 where=bulanan.values >= 0, color='#2d6a4f', alpha=0.15)
ax2.fill_between(bulanan.index, bulanan.values, 0,
                 where=bulanan.values < 0, color='#9d0208', alpha=0.15)
ax2.set_title('(b) Rata-rata Skor Sentimen Bulanan — 6 Emiten', fontweight='bold')
ax2.set_ylabel('Skor Sentimen [-1, +1]')
ax2.grid(alpha=0.3, linestyle='--'); ax2.spines[['top', 'right']].set_visible(False)

fig.suptitle('Gambar 4.X Hasil Ekstraksi Sentimen IndoBERT\n'
             f'Emiten Sektor Energi LQ45 | April 2023 – Mei 2026',
             fontweight='bold', y=1.04)
plt.tight_layout()
path_g = DIR_GAMBAR / 'gambar_4_x_hasil_sentimen.png'
plt.savefig(path_g, dpi=200, bbox_inches='tight')
plt.show()
print(f'  ✅ Gambar disimpan: {path_g}')

---
## 💾 CELL 8 — Simpan Output Final

In [ ]:
KOLOM_ARTIKEL = ['Sumber', 'Kode_Emiten', 'Tanggal', 'Headline',
                 'Skor_Sentimen', 'Label_Sentimen', 'Confidence', 'Link']
df_out = df[[k for k in KOLOM_ARTIKEL if k in df.columns]].copy()
df_out['Tanggal'] = df_out['Tanggal'].dt.strftime('%Y-%m-%d')

path_artikel = DIR_FINAL / 'sentimen_artikel_FINAL.csv'
path_harian  = DIR_FINAL / 'sentimen_harian_per_emiten.csv'
df_out.to_csv(path_artikel, index=False)
df_harian.to_csv(path_harian, index=False)

print('=' * 70)
print('  ✅ SELESAI — EKSTRAKSI SENTIMEN INDOBERT')
print('=' * 70)
print(f'  Skor per artikel  : {path_artikel.name}  ({len(df_out):,} baris)')
print(f'  Fitur harian X2   : {path_harian.name}  ({len(df_harian):,} baris)')
print()
print('  📌 LANGKAH SELANJUTNYA:')
print('     → Merge sentimen_harian_per_emiten.csv dengan fitur teknikal')
print('       (PREP) memakai kunci [Kode_Emiten, Tanggal], lalu jalankan')
print('       MODEL_xgboost_vv2.ipynb dan MODEL_lstm_v2.ipynb.')
print('=' * 70)